# Pump-probe fluence & time resolution (grazing incidence)

Two small calculators for planning grazing-incidence pump-probe experiments,
ported from [`calx.pumpprobe`](https://github.com/htlemke/calx) (pure
numpy/scipy, no facility-specific dependencies, so reimplemented directly in
`bernina_prep.timing` rather than depending on the full `calx` package).

Each parameter row has a small live plot of the output versus *that*
parameter, swept over whatever range you've actually explored so far (it
grows as you move the slider/field further); a side-view diagram below each
calculator shows the beam(s) hitting the sample surface at the current
angles, both updating automatically.

In [ ]:
import sys
from pathlib import Path

try:
    import bernina_prep  # noqa: F401
except ImportError:
    sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

from bernina_prep.timing import fluence, grazing_incidence_time_resolution

plt.ioff()
STYLE = {"description_width": "initial"}
INPUT_W = widgets.Layout(width="150px")
PLOT_W, PLOT_H = 3.0, 1.6

# Shared between both calculators below: true-to-scale 1:1 diagrams are
# barely visible at the shallow angles typical here, so default to a
# "visible" (auto, non-equal) aspect and let it be toggled per diagram.
aspect_toggle = widgets.ToggleButtons(
    options=[("visible (exaggerated)", "visible"), ("1:1 (true scale)", "equal")],
    value="visible",
    description="aspect ratio",
    style=STYLE,
)


def sensitivity_panel(param_specs, compute_fn, value_fmt, output_unit=""):
    """Build an interactive panel: one row per parameter (label + input +
    small live sensitivity plot), a result line, and wiring so that any
    change recomputes the result and every plot.

    param_specs: list of (name, widget, unit_label) -- widget's .value is
    used as the live parameter value; its historical min/max (seeded from
    its initial value) defines the plot's x-range, growing as you explore.
    compute_fn: function(**{name: value}) -> float
    value_fmt: function(result) -> str, for the result line
    output_unit: unit label for the plots' shared y-axis (the output value)
    """
    names = [n for n, _, _ in param_specs]
    widgets_by_name = {n: w for n, w, _ in param_specs}
    units = {n: u for n, _, u in param_specs}
    explored_range = {n: [w.value, w.value] for n, w, _ in param_specs}

    plot_outs = {n: widgets.Output() for n in names}
    result_out = widgets.Output()

    def current_kwargs():
        return {n: widgets_by_name[n].value for n in names}

    def refresh_plot(name):
        lo, hi = explored_range[name]
        span = hi - lo
        if span < 1e-12:
            lo, hi = lo - max(abs(lo) * 0.1, 1e-6), hi + max(abs(hi) * 0.1, 1e-6)
        else:
            lo, hi = lo - span * 0.1, hi + span * 0.1
        xs = np.linspace(lo, hi, 40)
        base = current_kwargs()
        ys = []
        for x in xs:
            kw = dict(base)
            kw[name] = x
            try:
                ys.append(compute_fn(**kw))
            except Exception:
                ys.append(np.nan)
        fig, ax = plt.subplots(figsize=(PLOT_W, PLOT_H))
        ax.plot(xs, ys, lw=1.2, color="#2166ac")
        ax.axvline(base[name], color="#d62728", lw=0.8, ls="--")
        unit_suffix = f" [{units[name]}]" if units[name] else ""
        ax.set_xlabel(f"{name}{unit_suffix}", fontsize=8)
        ax.set_ylabel(output_unit, fontsize=8)
        ax.tick_params(labelsize=7)
        ax.margins(x=0)
        fig.tight_layout(pad=0.3)
        with plot_outs[name]:
            plot_outs[name].clear_output(wait=True)
            display(fig)
        plt.close(fig)

    def refresh_all(_=None):
        with result_out:
            result_out.clear_output(wait=True)
            try:
                print(value_fmt(compute_fn(**current_kwargs())))
            except Exception as e:
                print(str(e))
        for n in names:
            refresh_plot(n)

    def on_change(change, name):
        v = change["new"]
        explored_range[name][0] = min(explored_range[name][0], v)
        explored_range[name][1] = max(explored_range[name][1], v)
        refresh_all()

    for n, w, _ in param_specs:
        w.observe(lambda change, n=n: on_change(change, n), names="value")

    rows = []
    for n, w, u in param_specs:
        row = widgets.HBox(
            [w, plot_outs[n]],
            layout=widgets.Layout(align_items="center"),
        )
        rows.append(row)

    refresh_all()
    return widgets.VBox(rows + [result_out])


def draw_beam(ax, angle_deg, half_width, color, label=None, height=None):
    """Schematic: a COLLIMATED (parallel-sided) beam of half-width
    `half_width` [mm] converging onto the surface (y=0 line) at
    `angle_deg` from the surface, coming from the +x side, shaded at
    alpha=0.5.

    `height` [mm] is how high above the surface to draw it -- deliberately
    NOT tied to the footprint length (default: a few beam-widths instead).
    At a few degrees grazing incidence the footprint is already tens of mm
    long; scaling height off *that* makes shallower beams draw as an
    ever-thinner sliver rather than a comparably-sized wedge, which is the
    opposite of what a "make it visible" mode should do. Comparing two
    beams of very different angles this way (same height, whatever x-extent
    each needs to reach it) directly shows which is shallower, rather than
    both just looking equally invisible.

    Returns (footprint_length_mm, x_extent_mm) -- x_extent is how far in x
    the drawn shape reaches, so callers can size the plot to fit it.
    """
    angle = np.deg2rad(max(angle_deg, 0.2))
    footprint = 2 * half_width / np.sin(angle)
    if height is None:
        height = 3 * half_width
    length = height / np.sin(angle)
    dirv = np.array([np.cos(angle), np.sin(angle)])
    x0 = half_width / np.sin(angle)
    p_bottom_pos = np.array([x0, 0.0])
    p_bottom_neg = np.array([-x0, 0.0])
    p_top_pos = p_bottom_pos + length * dirv
    p_top_neg = p_bottom_neg + length * dirv
    poly = np.array([p_bottom_neg, p_bottom_pos, p_top_pos, p_top_neg])
    ax.fill(poly[:, 0], poly[:, 1], color=color, alpha=0.5, edgecolor=color, linewidth=1.2)
    if label:
        top_mid = (p_top_pos + p_top_neg) / 2
        ax.annotate(label, top_mid, textcoords="offset points", xytext=(0, 4),
                    fontsize=8, color=color, ha="center")
    x_extent = max(abs(p_top_pos[0]), abs(p_top_neg[0]), x0)
    return footprint, height, x_extent


def apply_aspect(ax):
    if aspect_toggle.value == "equal":
        ax.set_aspect("equal", adjustable="box")
    else:
        ax.set_aspect("auto")


## Absorbed fluence

Pulse energy spread over the beam's footprint on the sample surface. At
grazing incidence (`angle` well below 90&deg;) the same beam energy is
spread over a much larger illuminated area than the beam's own
cross-section, reducing the fluence per unit area accordingly.

In [ ]:
fluence_specs = [
    ("energy_uj", widgets.FloatText(value=100.0, description="pulse energy", style=STYLE, layout=INPUT_W), "\u00b5J"),
    ("area_mm2", widgets.FloatText(value=0.01, step=0.001, description="beam area", style=STYLE, layout=INPUT_W), "mm\u00b2"),
    ("angle_deg", widgets.FloatSlider(value=90.0, min=0.1, max=90.0, step=0.1, description="incidence angle", style=STYLE, layout=widgets.Layout(width="260px")), "deg"),
    ("reflectivity", widgets.FloatSlider(value=0.0, min=0.0, max=0.99, step=0.01, description="reflectivity", style=STYLE, layout=widgets.Layout(width="260px")), ""),
]


def fluence_value(energy_uj, area_mm2, angle_deg, reflectivity):
    return fluence(energy_uj * 1e-6, area_mm2 * 1e-6, np.deg2rad(angle_deg), reflectivity)


fluence_widgets = {n: w for n, w, _ in fluence_specs}
display(sensitivity_panel(
    fluence_specs, fluence_value,
    lambda f: f"Absorbed fluence: {f:.4f} mJ/cm\u00b2",
    output_unit="fluence [mJ/cm\u00b2]",
))


### Beam footprint (side view)

Sample surface as a horizontal line; the beam (shaded, collimated/parallel
-- not converging) hits it at the current incidence angle. Since this
calculator only takes a beam *area*, the diagram assumes a circular beam
and uses its diameter as the effective linear size (labeled explicitly
below) -- purely a drawing convenience, it doesn't affect the fluence
number above.

In [ ]:
fluence_diagram_out = widgets.Output()


def refresh_fluence_diagram(_=None):
    with fluence_diagram_out:
        fluence_diagram_out.clear_output(wait=True)
        angle = fluence_widgets["angle_deg"].value
        area = fluence_widgets["area_mm2"].value
        d_eff = 2 * np.sqrt(area / np.pi)  # circular-equivalent diameter [mm]
        fig, ax = plt.subplots(figsize=(5, 2.8))
        ax.axhline(0, color="#333", lw=1.5)
        fp, height, x_extent = draw_beam(ax, angle, half_width=d_eff / 2, color="#d62728", label=f"{angle:.1f}\u00b0")
        apply_aspect(ax)
        ax.set_xlim(-x_extent * 1.15, x_extent * 1.15)
        ax.set_ylim(-0.05 * height, 1.15 * height)
        ax.set_xlabel("along surface [mm]", fontsize=8)
        ax.set_ylabel("height above surface [mm]", fontsize=8)
        ax.set_title(f"footprint \u2248 {fp:.3f} mm (circular-equivalent \u2300 {d_eff:.4f} mm)", fontsize=9)
        ax.tick_params(labelsize=7)
        fig.tight_layout(pad=0.3)
        display(fig)
    plt.close(fig)


for _n, _w, _ in fluence_specs:
    if _n in ("angle_deg", "area_mm2"):
        _w.observe(refresh_fluence_diagram, names="value")
aspect_toggle.observe(refresh_fluence_diagram, names="value")
refresh_fluence_diagram()
display(widgets.VBox([aspect_toggle, fluence_diagram_out]))


## Geometric time resolution at grazing incidence

Independent of the pump/probe pulse durations themselves, a **purely
geometric** timing smear arises whenever pump and probe cross the sample
surface at an angle to each other with a finite footprint: different points
across that footprint see the pump wavefront arrive at slightly different
times, because of the path-length difference across the (often strongly
elongated, at grazing incidence) illuminated area.

The sign of the result indicates *which* beam's wavefront leads across the
footprint (pump vs. probe); the **magnitude** is the relevant smearing
contribution to add (in quadrature with the pulse durations and other
jitter sources) to get the overall time resolution.

In [ ]:
timeres_specs = [
    ("probe_size_mm", widgets.FloatText(value=0.5, step=0.01, description="probe size", style=STYLE, layout=INPUT_W), "mm"),
    ("probe_angle_deg", widgets.FloatText(value=0.5, step=0.05, description="probe angle", style=STYLE, layout=INPUT_W), "deg"),
    ("pump_angle_deg", widgets.FloatText(value=5.0, step=0.1, description="pump angle", style=STYLE, layout=INPUT_W), "deg"),
]


def timeres_value(probe_size_mm, probe_angle_deg, pump_angle_deg):
    return grazing_incidence_time_resolution(
        probe_size_mm * 1e-3, np.deg2rad(probe_angle_deg), np.deg2rad(pump_angle_deg)
    ) * 1e15


timeres_widgets = {n: w for n, w, _ in timeres_specs}
display(sensitivity_panel(
    timeres_specs, timeres_value,
    lambda t: f"Geometric time smear: {t:+.2f} fs  (magnitude: {abs(t):.2f} fs)",
    output_unit="time smear [fs]",
))


### Pump/probe beam intersection on the sample (side view)

Both beams (shaded, collimated) on the same surface line: the probe (blue,
typically much more grazing) and the pump (orange, typically steeper).
Their footprints rarely line up end-to-end -- that mismatch is exactly the
geometric smear computed above.

In [ ]:
timeres_diagram_out = widgets.Output()


def refresh_timeres_diagram(_=None):
    with timeres_diagram_out:
        timeres_diagram_out.clear_output(wait=True)
        probe_size = timeres_widgets["probe_size_mm"].value
        probe_angle = timeres_widgets["probe_angle_deg"].value
        pump_angle = timeres_widgets["pump_angle_deg"].value
        half_w = probe_size / 2
        fig, ax = plt.subplots(figsize=(5, 3))
        ax.axhline(0, color="#333", lw=1.5)
        # Both beams share the same (angle-independent) drawn height, so
        # whichever is shallower simply needs more x-range to reach it --
        # directly showing the angle mismatch rather than both beams just
        # looking equally flat.
        fp_probe, height, x_probe = draw_beam(ax, probe_angle, half_width=half_w, color="#2166ac", label=f"probe {probe_angle:.2f}\u00b0")
        fp_pump, _, x_pump = draw_beam(ax, pump_angle, half_width=half_w, color="#e6550d", label=f"pump {pump_angle:.2f}\u00b0", height=height)
        apply_aspect(ax)
        x_extent = max(x_probe, x_pump)
        ax.set_xlim(-x_extent * 1.15, x_extent * 1.15)
        ax.set_ylim(-0.05 * height, 1.15 * height)
        ax.set_xlabel("along surface [mm]", fontsize=8)
        ax.set_ylabel("height above surface [mm]", fontsize=8)
        t_fs = timeres_value(probe_size, probe_angle, pump_angle)
        ax.set_title(f"geometric smear: {t_fs:+.1f} fs", fontsize=9)
        ax.tick_params(labelsize=7)
        fig.tight_layout(pad=0.3)
        display(fig)
    plt.close(fig)


for _n, _w, _ in timeres_specs:
    _w.observe(refresh_timeres_diagram, names="value")
aspect_toggle.observe(refresh_timeres_diagram, names="value")
refresh_timeres_diagram()
display(widgets.VBox([aspect_toggle, timeres_diagram_out]))
